In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/kagglethon-1-0/dish_ingredient_matrix.csv
/kaggle/input/kagglethon-1-0/sample_submission.csv
/kaggle/input/kagglethon-1-0/public_train.csv
/kaggle/input/kagglethon-1-0/grocery_prices.csv
/kaggle/input/kagglethon-1-0/test.csv


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
DATA_PATH = "/kaggle/input/kagglethon-1-0/"

train = pd.read_csv(DATA_PATH + "public_train.csv")
test  = pd.read_csv(DATA_PATH + "test.csv")


/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


In [3]:
prep_cols = [c for c in train.columns if c.startswith("prepared_qty_")]
sold_cols = [c for c in train.columns if c.startswith("sold_qty_")]

print(len(prep_cols), "prepared columns")
print(len(sold_cols), "sold columns")
for df in [train, test]:
    df["total_prepared"] = df[prep_cols].sum(axis=1)
    df["total_sold"] = df[sold_cols].sum(axis=1)
    df["naive_waste"] = (df["total_prepared"] - df["total_sold"]).clip(lower=0)
def nrmse(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return rmse / (y_true.max() - y_true.min())

baseline_score = nrmse(train["total_waste"], train["naive_waste"])
print("Naive baseline NRMSE:", baseline_score)
train["residual"] = train["total_waste"] - train["naive_waste"]


53 prepared columns
53 sold columns
Naive baseline NRMSE: 0.046857064458340446


In [4]:
features = [
    "is_weekend",
    "is_religious_holiday",
    "sport_event_phase",
    "total_prepared",
    "total_sold"
]
le = LabelEncoder()
train["sport_event_phase"] = le.fit_transform(train["sport_event_phase"].astype(str))
test["sport_event_phase"] = le.transform(test["sport_event_phase"].astype(str))


In [5]:
X = train[features]
y = train["residual"]

model = lgb.LGBMRegressor(
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

model.fit(X, y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001555 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 459
[LightGBM] [Info] Number of data points in the train set: 3652, number of used features: 5
[LightGBM] [Info] Start training from score 0.926424


LGBMRegressor(learning_rate=0.05, n_estimators=600, random_state=42)

In [6]:
test["residual_pred"] = model.predict(test[features])

test["final_waste"] = (
    test["naive_waste"] + test["residual_pred"]
).clip(lower=0)
submission = pd.read_csv(DATA_PATH + "sample_submission.csv")
submission["total_waste"] = test["final_waste"]
submission.to_csv("submission.csv", index=False)

submission.head()


,id,total_waste
0,0,109.864033
1,1,114.403426
2,2,111.582888
3,3,117.744854
4,4,111.517236


In [7]:
# Evaluate residual model on train
train["residual_pred"] = model.predict(train[features])

train["final_pred"] = (
    train["naive_waste"] + train["residual_pred"]
).clip(lower=0)

final_score = nrmse(train["total_waste"], train["final_pred"])
print("Final model NRMSE:", final_score)


Final model NRMSE: 0.03675882612380488


In [8]:
for df in [train, test]:
    df["prep_to_sale_ratio"] = df["total_prepared"] / (df["total_sold"] + 1)
    df["overprep_flag"] = (df["total_prepared"] > df["total_sold"]).astype(int)
features = [
    "total_prepared",
    "total_sold",
    "imbalance_ratio",
    "prep_to_sale_ratio",
    "overprep_flag",
    "is_weekend",
    "is_religious_holiday",
    "sport_event_phase"
]

monotone_constraints = [
    0,  # is_weekend
    0,  # holiday
    0,  # sport_event
    1,  # total_prepared ↑ waste
    -1, # total_sold ↓ waste
    1,  # imbalance_ratio ↑ waste
    1,  # prep_to_sale_ratio ↑ waste
    1   # overprep_flag ↑ waste
]


In [9]:
# imbalance ratio feature
train["imbalance_ratio"] = (
    train["total_prepared"] - train["total_sold"]
) / (train["total_prepared"] + 1)

test["imbalance_ratio"] = (
    test["total_prepared"] - test["total_sold"]
) / (test["total_prepared"] + 1)


In [10]:
print("imbalance_ratio" in train.columns)


True


In [11]:
train["prep_to_sale_ratio"] = train["total_prepared"] / (train["total_sold"] + 1)
test["prep_to_sale_ratio"]  = test["total_prepared"] / (test["total_sold"] + 1)

train["overprep_flag"] = (train["total_prepared"] > train["total_sold"]).astype(int)
test["overprep_flag"]  = (test["total_prepared"] > test["total_sold"]).astype(int)


In [12]:
features = [
    "total_prepared",
    "total_sold",
    "imbalance_ratio",
    "prep_to_sale_ratio",
    "overprep_flag",
    "is_weekend",
    "is_religious_holiday",
    "sport_event_phase"
]


In [13]:
model = lgb.LGBMRegressor(
    n_estimators=900,
    learning_rate=0.03,
    num_leaves=31,
    min_data_in_leaf=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(train[features], train["residual"])


[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000127 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 969
[LightGBM] [Info] Number of data points in the train set: 3652, number of used features: 7
[LightGBM] [Info] Start training from score 0.926424


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.03, min_data_in_leaf=50,
              n_estimators=900, random_state=42, subsample=0.8)

In [14]:
train["residual_pred"] = model.predict(train[features])
train["final_pred"] = (
    train["naive_waste"] + train["residual_pred"]
).clip(lower=0)

print("Final NRMSE:", nrmse(train["total_waste"], train["final_pred"]))


[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
Final NRMSE: 0.035922546750332914


In [15]:
test["residual_pred"] = model.predict(test[features])

test["final_pred"] = (
    test["naive_waste"] + test["residual_pred"]
).clip(lower=0)

submission = pd.read_csv("/kaggle/input/kagglethon-1-0/sample_submission.csv")
submission["total_waste"] = test["final_pred"]

submission.to_csv("submission.csv", index=False)


[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50


In [16]:
test["residual_pred"] = model.predict(test[features])

test["final_pred"] = (
    test["naive_waste"] + test["residual_pred"]
).clip(lower=0)

submission = pd.read_csv("/kaggle/input/kagglethon-1-0/sample_submission.csv")
submission["total_waste"] = test["final_pred"]

submission.to_csv("submission.csv", index=False)


[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50


In [17]:
train["error"] = train["final_pred"] - train["total_waste"]

train["error"].describe()


count    3.652000e+03
mean    -1.711931e-09
std      7.005856e+00
min     -2.751878e+01
25%     -4.575452e+00
50%      1.178828e-01
75%      4.543588e+00
max      3.028022e+01
Name: error, dtype: float64

In [18]:
import numpy as np

def nrmse(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return rmse / (y_true.max() - y_true.min())


In [19]:
final_nrmse = nrmse(train["total_waste"], train["final_pred"])
print("Final NRMSE:", final_nrmse)


Final NRMSE: 0.035922546750332914
